# E-Vaporiser Image Classifier (Transfer Learning)

An online classified-ads marketplace wants to **automatically flag listings whose photos show an e-vaporiser (vape)** so they can be blocked before going live. This notebook builds the model behind that feature: a binary image classifier that answers one question for any uploaded photo — **"is this a vape, yes or no?"**

**What this notebook does, end-to-end:**
1. **Data** — sources vape photos and hard-negative look-alikes (lighter, USB drive, power bank) from Roboflow, and everyday-object negatives from Imagenette, then splits into train/val/test.
2. **Augmentation** — applies and *justifies* image augmentations, and visualises them.
3. **Models** — compares **two backbones**:
   - **ResNet** from **torchvision** (a CNN)
   - **ViT** (Vision Transformer) from **Hugging Face** (a transformer)
4. **Strategies** — trains **each** backbone with **two strategies**:
   - **Full fine-tune** — update *all* layers
   - **Head-only** — freeze the backbone, train *only* the new classifier head
5. **Tracking** — logs every run to **Weights & Biases (wandb.ai)**.
6. **Comparison** — loss/accuracy curves, a test-accuracy table, and a confusion matrix per configuration.
7. **Selection & demo** — picks the **two best configurations** and launches a seller-portal-style Gradio app, right in this notebook, to compare them side-by-side on an uploaded photo.

> Run the cells **top to bottom**. A **GPU runtime is strongly recommended** (Colab: *Runtime → Change runtime type → GPU*).

**Configuration matrix (4 training runs):**

| | Full fine-tune | Head-only |
|---|---|---|
| **ResNet (torchvision)** | `resnet_full` | `resnet_head` |
| **ViT (Hugging Face)** | `vit_full` | `vit_head` |


## Step 0 — Install dependencies

We install everything up front. On Colab most of these are already present; the cell is safe to re-run.
- `torch`, `torchvision` — the ResNet backbone + training engine
- `transformers` — the Hugging Face ViT backbone
- `roboflow` — to download the vape and hard-negative datasets
- `wandb` — experiment tracking on wandb.ai
- `scikit-learn`, `seaborn`, `matplotlib` — metrics & plots
- `gradio` — powers the seller-portal demo in Step 12
- `qrcode[pil]` — turns the Step 12 public share link into a scannable QR code

In [ ]:
!pip install -q torch torchvision transformers roboflow wandb scikit-learn seaborn matplotlib gradio "qrcode[pil]"

## Step 1 — Imports and device

We pick the GPU automatically if one is available. Everything else is standard PyTorch.

In [ ]:
import os, copy, random, glob, shutil, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from transformers import ViTForImageClassification, ViTImageProcessor

from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_recall_fscore_support)

# Reproducibility: fixing the seed makes the train/val/test split and shuffling repeatable.
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
if device.type != 'cuda':
    print('WARNING: no GPU detected. Training (especially the ViT) will be slow. '
          'On Colab: Runtime -> Change runtime type -> GPU.')

## Step 2 — Log in to Weights & Biases (wandb.ai)

`wandb` records the loss/accuracy of every run to your online dashboard so the four configurations can be compared side-by-side.

1. Make a free account at https://wandb.ai and copy your API key from https://wandb.ai/authorize
2. Paste it when prompted below.

If you **don't** have a key, set `USE_WANDB = False` and the notebook still runs end-to-end (it just won't log online).

In [ ]:
import wandb

USE_WANDB = False          # set to False to skip online logging entirely
WANDB_PROJECT = 'evaporiser-classifier'

if USE_WANDB:
    try:
        wandb.login()      # paste your key when prompted (or set WANDB_API_KEY env var)
        print('wandb login OK ->', WANDB_PROJECT)
    except Exception as e:
        print('wandb login failed, continuing without online logging:', e)
        USE_WANDB = False
else:
    print('wandb disabled by USE_WANDB = False')

## Step 3 — Get the data

We build a **binary** dataset with two folders:

```
data/
  vape/       <- POSITIVE class: e-vaporiser photos (Roboflow)
  not_vape/   <- NEGATIVE class: everyday objects (Imagenette) + hard look-alikes (Roboflow)
```

| Image group | Source |
|---|---|
| **Positive — vape photos** | **Roboflow Universe** |
| **Easy negatives — everyday objects** | **Imagenette** (fast.ai, public URL, no login) |
| **Hard negatives — lighter / USB drive / power bank** | **Roboflow Universe** (one project per class) |

Vape photos and all three hard-negative classes come from Roboflow, so you'll need a free Roboflow API key
and a workspace/project/version for each of those four categories. Everyday-object negatives need no account
at all — Imagenette downloads straight from a public URL.

> **Finding the three hard-negative projects.** Go to https://universe.roboflow.com, search "lighter",
> "usb flash drive", and "power bank" in turn, open a project you like for each, and copy its **workspace**,
> **project name**, and **version number** from the project page into `RF_HARD_NEGATIVES` below (the same way
> `vape-dataset/vape-dataset` was found for the positive class).

#### Configuration — edit these, then run the download cells

In [ ]:
# ----- EDIT ME -----------------------------------------------------------
ROBOFLOW_API_KEY = ''          # <-- paste your Roboflow API key here (used for every Roboflow download below)

RF_VAPE = {'workspace': 'vape-dataset', 'project': 'vape-dataset', 'version': 1}   # positive class

RF_HARD_NEGATIVES = {                                   # <-- fill these in (see markdown above)
    'lighter':    {'workspace': '', 'project': '', 'version': 1},
    'usb_drive':  {'workspace': '', 'project': '', 'version': 1},
    'power_bank': {'workspace': '', 'project': '', 'version': 1},
}

MAX_PER_CLASS     = 800        # cap vape images to keep training fast & balanced
HARD_NEG_FRACTION = 0.35       # share of the negative class that should be 'hard' look-alikes
# -------------------------------------------------------------------------

DATA_ROOT    = '/content/data'    # final binary dataset lives here
VAPE_DIR     = os.path.join(DATA_ROOT, 'vape')
NOT_VAPE_DIR = os.path.join(DATA_ROOT, 'not_vape')
os.makedirs(VAPE_DIR, exist_ok=True)
os.makedirs(NOT_VAPE_DIR, exist_ok=True)

IMG_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

def collect_images(src_root):
    # Recursively list every image file under a folder.
    files = []
    for dirpath, _, filenames in os.walk(src_root):
        for fn in filenames:
            if fn.lower().endswith(IMG_EXTS):
                files.append(os.path.join(dirpath, fn))
    return files

def copy_sample(files, dst_dir, max_n):
    # Copy up to max_n images into dst_dir with unique names (safe to call more than once on the same folder).
    random.shuffle(files)
    start = len(collect_images(dst_dir))   # continue numbering so repeated calls don't overwrite each other
    n = 0
    for src in files:
        if n >= max_n:
            break
        try:
            Image.open(src).convert('RGB')                          # verify it actually opens
            shutil.copy(src, os.path.join(dst_dir, f'{start+n:05d}.jpg'))
            n += 1
        except Exception:
            pass                                                     # skip corrupt files
    return n

def download_roboflow(workspace, project_name, version_num, export_format='folder'):
    # Downloads one Roboflow Universe project and returns the local folder it was extracted to.
    # Returns None (and the caller just skips it) if workspace/project are blank or the download fails.
    if not workspace or not project_name:
        return None
    from roboflow import Roboflow
    try:
        version = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(workspace).project(project_name).version(version_num)
        try:
            return version.download(export_format).location
        except Exception:
            return version.download('voc').location          # some projects only offer detection-style exports
    except Exception as e:
        print(f'    could not download {workspace}/{project_name}: {e}')
        return None

#### 3a. Vape photos — the positive class

In [ ]:
if not ROBOFLOW_API_KEY:
    raise ValueError('Please paste your ROBOFLOW_API_KEY in the config cell above.')

loc = download_roboflow(RF_VAPE['workspace'], RF_VAPE['project'], RF_VAPE['version'])
vape_files = collect_images(loc) if loc else []
print('Found', len(vape_files), 'vape images in the download.')
n_vape = copy_sample(vape_files, VAPE_DIR, MAX_PER_CLASS)
print('Copied', n_vape, 'images into', VAPE_DIR)

#### 3b. Easy negatives — everyday objects (Imagenette)

In [ ]:
def get_easy_negatives(dst_dir, target_n):
    # Everyday-object photos from Imagenette (fast.ai, public URL, no login needed).
    import urllib.request, tarfile
    if not os.path.isdir('/content/imagenette2-160'):
        print('  downloading Imagenette...')
        url = 'https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz'
        urllib.request.urlretrieve(url, '/content/imagenette.tgz')
        with tarfile.open('/content/imagenette.tgz') as tf:
            tf.extractall('/content/')
    files = collect_images('/content/imagenette2-160')
    print(f'  Imagenette: {len(files)} images available across 10 categories')
    return copy_sample(files, dst_dir, target_n)

neg_target  = n_vape                              # keep the two classes balanced
hard_target = int(neg_target * HARD_NEG_FRACTION)
easy_target = neg_target # - hard_target          # hard negatives are omitted for now to keep classes balanced

n_easy = get_easy_negatives(NOT_VAPE_DIR, easy_target)
print(f'Easy negatives copied: {n_easy}')

#### 3c. Hard negatives — lighter / USB drive / power bank (Roboflow)

In [ ]:
per_class_target = hard_target // len(RF_HARD_NEGATIVES)

for label, cfg in RF_HARD_NEGATIVES.items():
    loc = download_roboflow(cfg['workspace'], cfg['project'], cfg['version'])
    if loc is None:
        print(f'  {label}: skipped -- fill in RF_HARD_NEGATIVES["{label}"] with a Roboflow workspace/project.')
        continue
    files = collect_images(loc)
    n = copy_sample(files, NOT_VAPE_DIR, per_class_target)
    print(f'  {label}: copied {n} images (found {len(files)} in the download)')

print('\nFinal counts:')
print('  vape     :', len(collect_images(VAPE_DIR)))
print('  not_vape :', len(collect_images(NOT_VAPE_DIR)))

## Step 4 — Build the dataset and the train / val / test split

`CLASS_NAMES = ['not_vape', 'vape']` so label **1 = vape** (the thing we want to catch).

We do a **stratified 70 / 15 / 15** split so all three sets keep the same vape:not-vape ratio.
Splitting on the *file list* (before any transforms) guarantees the test images are never seen during training.

In [ ]:
CLASS_NAMES = ['not_vape', 'vape']        # index 0 and 1
LABEL2NAME = {i: n for i, n in enumerate(CLASS_NAMES)}

# Build (path, label) pairs
all_samples = []
for label, name in enumerate(CLASS_NAMES):
    for f in collect_images(os.path.join(DATA_ROOT, name)):
        all_samples.append((f, label))
print('Total images:', len(all_samples))

# Stratified split: split each class separately, then merge
def split_indices(samples, train=0.70, val=0.15):
    by_class = {}
    for i, (_, lab) in enumerate(samples):
        by_class.setdefault(lab, []).append(i)
    tr, va, te = [], [], []
    for lab, idxs in by_class.items():
        random.shuffle(idxs)
        n = len(idxs); n_tr = int(n*train); n_va = int(n*val)
        tr += idxs[:n_tr]; va += idxs[n_tr:n_tr+n_va]; te += idxs[n_tr+n_va:]
    random.shuffle(tr); random.shuffle(va); random.shuffle(te)
    return tr, va, te

train_idx, val_idx, test_idx = split_indices(all_samples)
print(f'train {len(train_idx)} | val {len(val_idx)} | test {len(test_idx)}')

class VapeDataset(Dataset):
    # Returns (transformed_image, label). One dataset object per split/transform.
    def __init__(self, samples, indices, transform):
        self.items = [samples[i] for i in indices]
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label

## Step 5 — Data augmentation (and why)

**Augmentation** randomly perturbs each *training* image every epoch (flips, crops, colour jitter, rotation).
The model therefore never sees the exact same picture twice, which:
- **reduces over-fitting** — important here because our dataset is small;
- **mimics real marketplace photos** — sellers shoot vapes at odd angles, distances, lighting and crops, so the
  training data should reflect that variety.

**Validation and test images are *not* augmented** — we only resize/centre-crop/normalise them, because at
evaluation time we want a clean, deterministic measurement.

Two backbones expect **different normalisation**:
- **ResNet (ImageNet)** → mean `[0.485,0.456,0.406]`, std `[0.229,0.224,0.225]`
- **ViT (`google/vit-base-patch16-224`)** → the processor uses mean/std `[0.5,0.5,0.5]`

So we build a small dictionary of transforms per backbone.

In [ ]:
IMG_SIZE = 224

NORM = {
    'resnet': dict(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    'vit':    dict(mean=[0.5, 0.5, 0.5],       std=[0.5, 0.5, 0.5]),
}

def make_transforms(backbone):
    m, s = NORM[backbone]['mean'], NORM[backbone]['std']
    train_t = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),  # random zoom/crop
        transforms.RandomHorizontalFlip(),                         # left/right flip
        transforms.RandomRotation(15),                             # small tilt
        transforms.ColorJitter(0.2, 0.2, 0.2),                     # lighting/colour variation
        transforms.ToTensor(),
        transforms.Normalize(m, s),
    ])
    eval_t = transforms.Compose([                                  # clean, deterministic
        transforms.Resize(256),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(m, s),
    ])
    return train_t, eval_t

def make_loaders(backbone, batch_size=32):
    train_t, eval_t = make_transforms(backbone)
    ds_tr = VapeDataset(all_samples, train_idx, train_t)
    ds_va = VapeDataset(all_samples, val_idx,   eval_t)
    ds_te = VapeDataset(all_samples, test_idx,  eval_t)
    return {
        'train': DataLoader(ds_tr, batch_size=batch_size, shuffle=True,  num_workers=2),
        'val':   DataLoader(ds_va, batch_size=batch_size, shuffle=False, num_workers=2),
        'test':  DataLoader(ds_te, batch_size=batch_size, shuffle=False, num_workers=2),
    }

#### Visual check — what the augmented training images look like
Seeing the augmentation is the easiest way to confirm it is sensible (not so aggressive that the vape becomes
unrecognisable).

In [ ]:
train_t, _ = make_transforms('resnet')
inv_mean = np.array(NORM['resnet']['mean']); inv_std = np.array(NORM['resnet']['std'])

def show_tensor(ax, t):
    img = t.permute(1, 2, 0).numpy() * inv_std + inv_mean   # un-normalise for display
    ax.imshow(np.clip(img, 0, 1)); ax.axis('off')

sample_path, sample_label = all_samples[train_idx[0]]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle(f'8 augmented versions of one "{LABEL2NAME[sample_label]}" image', fontsize=13)
for ax in axes.ravel():
    show_tensor(ax, train_t(Image.open(sample_path).convert('RGB')))
plt.tight_layout(); plt.show()

## Step 6 — Build the two backbones with two strategies each

Both backbones are **pretrained on ImageNet**. Transfer learning means we keep those learned visual features and
only re-teach the model our 2-class decision.

**Head modification (required):**
- **ResNet** ends in a fully-connected layer `fc` of size `(512 → 1000)`. We replace it with `(512 → 2)`.
- **ViT** ends in a `classifier` layer `(768 → 1000)`. We replace it with `(768 → 2)`.

**The two strategies:**
- **`full`** — every layer is trainable; the whole network adapts to vapes.
- **`head`** — we **freeze** the backbone (`requires_grad = False`) and train **only** the new head. This is
  faster, needs less data, and is the classic "feature-extractor" approach.

To keep one training loop for both libraries, we wrap the ViT so it returns plain **logits** (a tensor), exactly
like the ResNet does.

In [ ]:
NUM_CLASSES = 2
VIT_CHECKPOINT = 'google/vit-base-patch16-224'

class ViTWrapper(nn.Module):
    # Makes the HuggingFace ViT behave like a torchvision model: forward(x) -> logits tensor.
    def __init__(self, vit):
        super().__init__()
        self.vit = vit
    def forward(self, x):
        return self.vit(pixel_values=x).logits

def build_model(backbone, strategy):
    # backbone in {'resnet','vit'}; strategy in {'full','head'}.
    if backbone == 'resnet':
        net = resnet18(weights=ResNet18_Weights.DEFAULT)        # pretrained CNN
        net.fc = nn.Linear(net.fc.in_features, NUM_CLASSES)     # new 2-class head
        if strategy == 'head':
            for p in net.parameters(): p.requires_grad = False  # freeze backbone
            for p in net.fc.parameters(): p.requires_grad = True# unfreeze head
        return net.to(device)

    elif backbone == 'vit':
        vit = ViTForImageClassification.from_pretrained(
            VIT_CHECKPOINT, num_labels=NUM_CLASSES,
            ignore_mismatched_sizes=True)                       # lets us swap the head
        if strategy == 'head':
            for p in vit.vit.parameters(): p.requires_grad = False   # freeze encoder
            for p in vit.classifier.parameters(): p.requires_grad = True
        return ViTWrapper(vit).to(device)

    raise ValueError(backbone)

# Each backbone needs its own normalised data loaders (built once, reused).
LOADERS = {'resnet': make_loaders('resnet'), 'vit': make_loaders('vit')}
print('Loaders ready for:', list(LOADERS.keys()))

## Step 7 — One training/evaluation loop for all runs

The loop is deliberately small and readable. It:
- runs one epoch of train or eval,
- tracks loss + accuracy,
- keeps the **best weights by validation accuracy**,
- logs every epoch to **wandb** (if enabled).

We use **class weights** in the loss so that if the two classes are slightly unbalanced the model still pays
attention to the vape class.

In [ ]:
def run_epoch(net, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    net.train() if is_train else net.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = net(x)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

def train_config(backbone, strategy, epochs=5, lr=None):
    name = f'{backbone}_{strategy}'
    print(f'\n===================  {name}  ===================')
    net = build_model(backbone, strategy)
    loaders = LOADERS[backbone]

    # head-only can use a higher LR (only a small layer trains); full fine-tune uses a gentle LR.
    if lr is None:
        lr = 1e-3 if strategy == 'head' else 3e-5 if backbone == 'vit' else 1e-4
    params = [p for p in net.parameters() if p.requires_grad]
    optimizer = optim.AdamW(params, lr=lr)
    criterion = nn.CrossEntropyLoss()

    run = None
    if USE_WANDB:
        run = wandb.init(project=WANDB_PROJECT, name=name, reinit=True,
                         config=dict(backbone=backbone, strategy=strategy,
                                     lr=lr, epochs=epochs))

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_acc, best_weights = 0.0, copy.deepcopy(net.state_dict())

    for epoch in range(epochs):
        tr_loss, tr_acc = run_epoch(net, loaders['train'], criterion, optimizer)
        va_loss, va_acc = run_epoch(net, loaders['val'],   criterion)
        history['train_loss'].append(tr_loss); history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc);   history['val_acc'].append(va_acc)
        print(f'  epoch {epoch+1}/{epochs}  '
              f'train_loss {tr_loss:.3f} acc {tr_acc:.3f} | val_loss {va_loss:.3f} acc {va_acc:.3f}')
        if USE_WANDB:
            wandb.log({'epoch': epoch+1, 'train_loss': tr_loss, 'val_loss': va_loss,
                       'train_acc': tr_acc, 'val_acc': va_acc})
        if va_acc > best_acc:
            best_acc = va_acc; best_weights = copy.deepcopy(net.state_dict())

    net.load_state_dict(best_weights)          # restore best epoch
    print(f'  best val_acc = {best_acc:.3f}')
    if USE_WANDB and run is not None:
        wandb.summary['best_val_acc'] = best_acc
        run.finish()
    return net, history, best_acc

## Step 8 — Train all four configurations

`EPOCHS` is small so the whole notebook finishes in a reasonable time on a Colab GPU. Increase it for better
accuracy. Head-only runs are fast; full fine-tuning (especially the ViT) is the slow part.

In [ ]:
EPOCHS = 5     # bump to 8-12 for stronger results if you have GPU time

CONFIGS = [('resnet', 'full'), ('resnet', 'head'),
           ('vit', 'full'),    ('vit', 'head')]

trained = {}     # name -> (net, history, best_val_acc)
for backbone, strategy in CONFIGS:
    net, hist, best = train_config(backbone, strategy, epochs=EPOCHS)
    trained[f'{backbone}_{strategy}'] = (net, hist, best)

print('\nDone. Trained configs:', list(trained.keys()))

## Step 9 — Compare the learning curves

Overlaying all four runs shows which configuration learns fastest and which generalises best
(val accuracy is the honest signal; training accuracy can be misleadingly high).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for name, (_, h, _) in trained.items():
    axes[0].plot(h['train_loss'], marker='o', label=name)
    axes[1].plot(h['val_loss'],   marker='o', label=name)
    axes[2].plot(h['val_acc'],    marker='o', label=name)
axes[0].set_title('Training loss')
axes[1].set_title('Validation loss')
axes[2].set_title('Validation accuracy')
for ax in axes:
    ax.set_xlabel('epoch'); ax.grid(True, alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## Step 10 — Test-set evaluation: accuracy table + confusion matrices

The test set was held out from training. For a content-moderation tool we care about more than raw accuracy:
- **Recall (vape)** — of all real vapes, how many did we catch? *Misses = banned items slipping through.*
- **Precision (vape)** — of everything we flagged, how many were really vapes? *Low precision annoys honest sellers.*

So we report accuracy **and** precision/recall/F1, plus a confusion matrix per configuration.

In [ ]:
@torch.no_grad()
def predict_all(net, loader):
    net.eval()
    ys, ps = [], []
    for x, y in loader:
        logits = net(x.to(device))
        ps.append(logits.argmax(1).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

rows = []
cms = {}
for name, (net, _, best_val) in trained.items():
    backbone = name.split('_')[0]
    y_true, y_pred = predict_all(net, LOADERS[backbone]['test'])
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[1], average='macro', zero_division=0)   # class 1 = vape
    rows.append((name, best_val, acc, pr, rc, f1))
    cms[name] = confusion_matrix(y_true, y_pred, labels=[0, 1])

# ----- accuracy table -----
print(f'{"config":16s} {"val_acc":>8s} {"test_acc":>9s} {"vape_P":>8s} {"vape_R":>8s} {"vape_F1":>8s}')
print('-'*62)
for name, bv, acc, pr, rc, f1 in rows:
    print(f'{name:16s} {bv:8.3f} {acc:9.3f} {pr:8.3f} {rc:8.3f} {f1:8.3f}')

# ----- confusion matrices -----
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, cm) in zip(axes, cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(name); ax.set_xlabel('predicted'); ax.set_ylabel('true')
plt.tight_layout(); plt.show()

## Step 11 — Select the two best configurations

We rank by **test accuracy** (you could rank by vape-recall instead for a stricter moderation policy) and keep
the **top two**. They're already sitting in memory from Step 8 — Step 12 uses them directly to launch the
seller-portal demo, all within this same Colab session (no files saved, no Google Drive needed).

In [ ]:
ranked = sorted(rows, key=lambda r: r[2], reverse=True)   # r[2] = test_acc
best_two = [r[0] for r in ranked[:2]]

print('Top-2 configurations:')
for name in best_two:
    test_acc = [r[2] for r in rows if r[0] == name][0]
    print(f'  {name}: test_acc = {test_acc:.3f}')

## Step 12 — Launch the seller-portal demo

This mimics the photo-upload step of a real classified-ad listing, using the same **seller-portal interface**
(hero banner, listing form, colour-coded verdict, and per-model probability cards) as the standalone
`seller_portal_app.py` / `seller_portal_webapp_colab.ipynb` deployments — running live, entirely within this
Colab session. Nothing is saved to disk or Google Drive; the two best models from Step 11 are used directly
from memory.

**This build adds three usability touches:**
- **Status bar above the photo** — as soon as a photo is checked, a coloured bar appears directly over the
  preview: **green "Looks fine"**, **amber "Pending human review"**, or **red "Blocked"**.
- **Mobile-responsive layout** — on a narrow phone screen the two columns collapse into one, and the listing-review
  panel drops to the **bottom**, below the Block threshold slider.
- **QR code** — the public share link Colab generates is also rendered as a QR code you can scan with a phone
  camera to open the app on your handset.

In [ ]:
import torch.nn.functional as F

class DemoModel:
    # In-memory equivalent of the standalone app's LoadedModel: same .predict()/.pretty_name/.name
    # interface, but wraps a model already trained in this session instead of a saved checkpoint.
    def __init__(self, name, net, transform, backbone, strategy):
        self.name = name
        self.net = net
        self.transform = transform
        self.backbone = backbone
        self.strategy = strategy
        self.vape_idx = CLASS_NAMES.index('vape') if 'vape' in CLASS_NAMES else len(CLASS_NAMES) - 1

    @torch.no_grad()
    def predict(self, pil_img):
        x = self.transform(pil_img.convert('RGB')).unsqueeze(0).to(device)
        probs = F.softmax(self.net(x), dim=1)[0].cpu().tolist()
        scores = {CLASS_NAMES[i]: float(probs[i]) for i in range(len(probs))}
        return scores, float(probs[self.vape_idx])

    @property
    def pretty_name(self):
        return f"{self.backbone.upper()} \u00b7 {self.strategy.replace('_', ' ')}"

demo_models = []
for name in best_two:
    net = trained[name][0]
    backbone, strategy = name.split('_')
    _, eval_t = make_transforms(backbone)
    demo_models.append(DemoModel(name, net, eval_t, backbone, strategy))

print('Demo ready with:', [m.name for m in demo_models])

In [ ]:
import gradio as gr

def verdict(vape_probs, threshold):
    # Combine the two models' vape probabilities into one moderation decision.
    flags = [p >= threshold for p in vape_probs]
    avg = sum(vape_probs) / len(vape_probs)
    if all(flags):
        return ("blocked", "Blocked — looks like an e-vaporiser",
                "Both models flagged this photo. The listing cannot be published.")
    if any(flags):
        return ("review", "Needs review — possible e-vaporiser",
                "The models disagree. Sent to a human reviewer before publishing.")
    return ("allowed", "Looks fine — no e-vaporiser detected",
            "Both models cleared this photo. The listing can be published.")

# The short label shown on the colour bar that sits directly above the photo preview.
STATUS_BAR = {
    'allowed': ('allowed', '✓  Looks fine'),
    'review':  ('review',  '⏳  Pending human review'),
    'blocked': ('blocked', '⛔  Blocked'),
}

def status_bar_html(state=None):
    # state None -> neutral "waiting" bar; otherwise green / amber / red per the verdict.
    if state is None:
        return "<div class='statusbar idle'>○  Awaiting photo</div>"
    cls, label = STATUS_BAR[state]
    return f"<div class='statusbar {cls}'>{label}</div>"

# Intentional palette: a calm trust-&-safety ops console.
# ink #15233B (deep slate) - paper #F7F8FA - line #E2E6EC - brand #2D6CDF
# clear #1F9D6B - review #C9820B - block #C2453C
CSS = '''
:root{
  --ink:#15233B; --muted:#5C6B82; --paper:#F7F8FA; --card:#FFFFFF;
  --line:#E2E6EC; --brand:#2D6CDF; --clear:#1F9D6B; --review:#C9820B; --block:#C2453C;
}
.gradio-container{background:var(--paper)!important; font-family:'Inter',system-ui,sans-serif;}
#hero{display:flex;align-items:center;gap:14px;padding:18px 22px;margin-bottom:6px;
  background:var(--ink);border-radius:14px;color:#fff;}
#hero .dot{width:34px;height:34px;border-radius:9px;background:var(--brand);
  display:flex;align-items:center;justify-content:center;font-size:18px;}
#hero h1{font-size:18px;margin:0;font-weight:650;letter-spacing:.2px;}
#hero p{margin:2px 0 0;font-size:12.5px;color:#AEBAD0;}
.panel{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:18px;}
.eyebrow{font-size:11px;font-weight:700;letter-spacing:.12em;text-transform:uppercase;
  color:var(--muted);margin-bottom:10px;}
/* colour bar directly above the photo preview: green=fine, amber=review, red=blocked */
.statusbar{border-radius:10px;padding:11px 14px;margin-bottom:10px;font-weight:650;
  font-size:13.5px;display:flex;align-items:center;gap:8px;border:1px solid transparent;
  letter-spacing:.2px;}
.statusbar.allowed{background:var(--clear);color:#fff;}
.statusbar.review {background:var(--review);color:#fff;}
.statusbar.blocked{background:var(--block);color:#fff;}
.statusbar.idle   {background:#EEF1F5;color:var(--muted);border-color:var(--line);}
.verdict{border-radius:12px;padding:16px 18px;margin-bottom:14px;border:1px solid transparent;}
.verdict h2{margin:0 0 4px;font-size:16px;color:inherit!important;}
.verdict p{margin:0;font-size:13px;opacity:.92;color:inherit!important;}
.verdict.allowed{background:#E9F7F1;border-color:#BFE7D6;color:#0E5C3F;}
.verdict.review {background:#FBF1DC;border-color:#F0DCA8;color:#7A5206;}
.verdict.blocked{background:#FAE9E7;border-color:#F0C5C0;color:#7E2A24;}
.modelcard{border:1px solid var(--line);border-radius:12px;padding:14px 16px;margin-bottom:12px;}
.modelcard .top{display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;}
.modelcard .nm{font-weight:650;color:var(--ink);font-size:14px;}
.modelcard .acc{font-size:11.5px;color:var(--muted);}
.bar{height:9px;border-radius:6px;background:#EEF1F5;overflow:hidden;}
.bar>span{display:block;height:100%;}
.barrow{display:flex;justify-content:space-between;font-size:12px;color:var(--muted);margin:9px 0 4px;}
.barrow b{color:var(--ink);}
.hint{font-size:12px;color:var(--muted);line-height:1.5;}
footer{display:none!important;}
/* Mobile: collapse the two columns into one. Source order puts the listing form
   (incl. the Block threshold slider) first and the review panel second, so on a
   narrow screen the review section stacks to the BOTTOM, after the slider. */
@media (max-width: 640px){
  #main-row{flex-direction:column!important;}
  #main-row > *{min-width:100%!important;flex:1 1 100%!important;}
  #review-col{order:2;}
  #listing-col{order:1;}
}
'''

HERO = '''
<div id='hero'>
  <div class='dot'>\U0001F6E1️</div>
  <div>
    <h1>Marketplace Listing Check</h1>
    <p>Prohibited-item screening · e-vaporiser detection · two-model comparison</p>
  </div>
</div>
'''

def build_interface(models):
    def render(img, threshold):
        if img is None:
            empty = "<div class='panel'><div class='eyebrow'>Listing review</div>" \
                    "<p class='hint'>Add a product photo to run the check.</p></div>"
            return status_bar_html(None), empty

        results = [m.predict(img) for m in models]
        vape_probs = [r[1] for r in results]
        state, title, detail = verdict(vape_probs, threshold)

        cards = ""
        for m, (scores, vp) in zip(models, results):
            conf = max(scores.values())
            top = max(scores, key=scores.get)
            bar_color = "var(--block)" if vp >= threshold else "var(--clear)"
            cards += f'''
            <div class='modelcard'>
              <div class='top'>
                <span class='nm'>{m.pretty_name}</span>
                <span class='acc'>{m.name}</span>
              </div>
              <div class='barrow'><span>vape probability</span><b>{vp*100:.1f}%</b></div>
              <div class='bar'><span style='width:{vp*100:.1f}%;background:{bar_color}'></span></div>
              <div class='barrow' style='margin-top:8px;'>
                <span>top label</span><b>{top} · {conf*100:.1f}% confidence</b></div>
            </div>'''

        agree = "agree" if (vape_probs[0] >= threshold) == (vape_probs[1] >= threshold) else "disagree"
        review_html = f'''
        <div class='panel'>
          <div class='eyebrow'>Listing review</div>
          <div class='verdict {state}'><h2>{title}</h2><p>{detail}</p></div>
          {cards}
          <p class='hint'>The two models <b style='color:var(--review);font-weight:700'>{agree}</b> at the current threshold
          ({threshold*100:.0f}%). Lower the threshold to catch more vapes
          (higher recall); raise it to reduce false alarms on honest listings.</p>
        </div>'''
        return status_bar_html(state), review_html

    with gr.Blocks(css=CSS, title="Marketplace Listing Check") as demo:
        gr.HTML(HERO)
        with gr.Row(elem_id="main-row"):
            with gr.Column(scale=1, elem_id="listing-col"):
                gr.HTML("<div class='eyebrow'>New listing</div>")
                gr.Textbox(label="Item title", placeholder="e.g. Vintage desk lamp",
                           value="")
                gr.Textbox(label="Price", placeholder="$0.00", value="")
                # colour-coded review-status bar sits directly above the photo preview
                status = gr.HTML(status_bar_html(None))
                photo = gr.Image(type="pil", label="Product photo",
                                 sources=["upload", "clipboard"], height=300)
                threshold = gr.Slider(0.05, 0.95, value=0.5, step=0.05,
                                      label="Block threshold (vape probability)")
                gr.HTML("<p class='hint'>Trust & Safety policy: photos likely to show an "
                        "e-vaporiser are blocked before the listing goes live.</p>")
            with gr.Column(scale=1, elem_id="review-col"):
                review = gr.HTML("<div class='panel'><div class='eyebrow'>Listing review</div>"
                                 "<p class='hint'>Add a product photo to run the check.</p></div>")

        photo.change(render, [photo, threshold], [status, review])
        threshold.change(render, [photo, threshold], [status, review])

    return demo

demo = build_interface(demo_models)

# launch() returns (fastapi_app, local_url, share_url); we keep share_url for the QR code below.
app, local_url, share_url = demo.launch(share=True, debug=False)

# --- QR code for the public link: scan with a phone camera to open the app ---
import qrcode
from IPython.display import display

if share_url:
    qr_img = qrcode.make(share_url).resize((260, 260))
    print('Scan this QR code with your phone camera to open the seller portal:')
    print(share_url)
    display(qr_img)
else:
    print('No public share URL was produced (share=True may be blocked on this network).')
    print('Local URL:', local_url)

## Step 13 — Summary

**What this notebook built.** An end-to-end pipeline that sources vape photos and two flavours of negative
examples (everyday objects and close look-alikes) from open, no-login sources, augments the training images,
and compares **ResNet (torchvision)** vs **ViT (Hugging Face)** under **full fine-tune** vs **head-only** — four
runs, all tracked on wandb. Each configuration was evaluated on a held-out test set (accuracy, vape
precision/recall, confusion matrices), and the **two best** configurations power a live seller-portal demo,
launched directly from this notebook.

**Honest limitations of this run:**
- **Sourced data may be noisy.** Datasets found via keyword search weren't curated for this exact
  task, so it's worth manually reviewing a sample of each class before trusting the numbers.
- **Class balance is approximate.** The hard-negative fraction is a configurable target, not a guarantee — check
  the printed counts from Step 3 to see what was actually collected.
- **Small dataset, few epochs.** Increase `MAX_PER_CLASS` and `EPOCHS` for a more convincing comparison if you
  have GPU time to spare.
- **Decision threshold.** The demo's slider defaults to 0.5; a real moderation policy would tune this toward
  catching more vapes, accepting a few false positives that a human reviewer can clear.